[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/patterns/crewai_patterns.ipynb)

# Seven matched patterns with CrewAI Flow

Each pattern is an independently routed CrewAI Flow run. The shared mock model—not CrewAI's default provider—proposes the same structured decisions used by the other notebooks. Mock runtime is about one minute on CPU; local inference is practical but slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import asyncio
import contextlib
import io
import json
import os
import tempfile
from pathlib import Path
from typing import ClassVar

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)
from crewai.flow.flow import Flow, listen, router, start  # noqa: E402
from pydantic import BaseModel, ConfigDict, Field  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import (  # noqa: E402
    CriticDecision,
    Message,
    MessageRole,
    PlanDecision,
    RouteDecision,
    ToolDefinition,
)

fixture_data = json.loads(
    (ROOT / "data/research_assistant/pattern_mock_scenarios.json").read_text()
)
catalogue = fixture_data["catalogue"]

## Shared contracts and CrewAI routing
The `@start`, `@router` and `@listen` decorators expose dispatch. Each listener contains its own decision, validation, execution, trace and stop logic. The Flow receives no effectful tools.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Claim(StrictModel):
    schema_id: ClassVar[str] = "tutorial.claim.v1"
    claim: str
    source_id: str
    supported: bool


class Answer(StrictModel):
    schema_id: ClassVar[str] = "tutorial.pattern_answer.v1"
    answer: str
    source_ids: tuple[str, ...]


class ParallelDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.parallel.v1"
    queries: tuple[str, ...] = Field(min_length=2)
    aggregation: str


class OrchestrationDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.orchestration.v1"
    assignments: tuple[dict, ...] = Field(min_length=1, max_length=2)


def model_for(name):
    return DeterministicMockClient(
        ScriptedScenarioFixture.model_validate(
            {
                "fixture_version": fixture_data["fixture_version"],
                "scenario": name,
                "provenance": fixture_data["provenance"],
                "responses": fixture_data["scenarios"][name],
            }
        )
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def search(query, limit=3):
    terms = set(query.casefold().split())
    return [r for r in catalogue if terms & set(" ".join(r["topics"]).casefold().split())][:limit]


search_tool = ToolDefinition(
    name="search_catalogue",
    description="Search bounded evidence.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}, "max_results": {"type": "integer"}},
        "required": ["query", "max_results"],
    },
)

In [ ]:
class MatchedPatternFlow(Flow):
    @start()
    def receive(self):
        return self.state["pattern"]

    @router(receive)
    def select(self, pattern):
        allowed = {
            "prompt_chaining",
            "routing",
            "parallelisation",
            "react",
            "planner_executor",
            "critic_reviser",
            "orchestrator_worker",
        }
        return pattern if pattern in allowed else "safe_fallback"

    @listen("prompt_chaining")
    async def run_chain(self):
        model = model_for("prompt_chaining")
        claim = await propose(model, Claim, "Extract one cited claim.")
        answer = await propose(model, Answer, f"S synthesise only {claim.model_dump()}")
        return {
            "answer": answer.model_dump(),
            "trace": ["extract", "synthesise"],
            "stop": "criteria_met",
        }

    @listen("routing")
    async def run_route(self):
        route = await propose(model_for("routing"), RouteDecision, "Route the research request.")
        valid = route.route in {"research", "clarify"}
        results = search("meal planning portion") if route.route == "research" else []
        return {
            "results": results,
            "trace": ["route_validated", route.route],
            "stop": "criteria_met" if valid else "safe_fallback",
        }

    @listen("parallelisation")
    async def run_parallel(self):
        decision = await propose(
            model_for("parallelisation"), ParallelDecision, "Propose two searches."
        )
        batches = await asyncio.gather(*(asyncio.to_thread(search, q, 2) for q in decision.queries))
        source_ids = sorted({r["source_id"] for batch in batches for r in batch})
        return {
            "source_ids": source_ids,
            "trace": ["fan_out", "validated_union"],
            "stop": "criteria_met",
        }

    @listen("react")
    async def run_react(self):
        model = model_for("react")
        response = await model.generate([user("Choose one bounded tool.")], tools=[search_tool])
        call = response.tool_calls[0]
        if call.name != "search_catalogue":
            return {"trace": ["rejected"], "stop": "invalid_tool"}
        records = search(call.arguments["query"], call.arguments["max_results"])
        return {
            "records": records,
            "steps": 1,
            "trace": ["tool_validated", "tool_executed"],
            "stop": "criteria_met",
        }

    @listen("planner_executor")
    async def run_plan(self):
        plan = await propose(model_for("planner_executor"), PlanDecision, "Plan bounded research.")
        valid = len(plan.steps) == 3 and all(
            plan.steps[i].depends_on == (plan.steps[i - 1].step_id,) for i in (1, 2)
        )
        return {
            "plan": plan.model_dump(),
            "trace": ["plan_validated", "executed"] if valid else ["dependency_stop"],
            "stop": "criteria_met" if valid else "invalid_plan",
        }

    @listen("critic_reviser")
    async def run_critic(self):
        model = model_for("critic_reviser")
        draft = await propose(model, Answer, "Draft.")
        critic = await propose(model, CriticDecision, f"Critique {draft.model_dump()}")
        revisions = 0
        if not critic.accepted:
            draft = await propose(model, Answer, f"Revise once: {critic.issues}")
            revisions = 1
        return {
            "answer": draft.model_dump(),
            "revisions": revisions,
            "trace": ["draft", "critic", "revision"],
            "stop": "criteria_met" if draft.source_ids else "revision_budget_exhausted",
        }

    @listen("orchestrator_worker")
    async def run_workers(self):
        model = model_for("orchestrator_worker")
        decision = await propose(model, OrchestrationDecision, "Assign two reviewers.")
        allowed = {"intervention_reviewer", "planning_reviewer"}
        if any(a["worker"] not in allowed for a in decision.assignments):
            return {"stop": "invalid_assignment"}
        batches = await asyncio.gather(
            *(asyncio.to_thread(search, a["query"], 2) for a in decision.assignments)
        )
        answer = await propose(model, Answer, f"S synthesise {batches}")
        return {
            "answer": answer.model_dump(),
            "worker_count": len(batches),
            "trace": ["assign", "workers", "synthesise"],
            "stop": "criteria_met",
        }

    @listen("safe_fallback")
    def reject_unknown(self):
        return {"trace": ["invalid_pattern"], "stop": "safe_fallback"}

## Execute and evaluate
The same Flow is invoked independently for each pattern. Output suppression hides CrewAI's decorative console panels, not the trace returned below. Limitations: Flow routing does not guarantee semantic correctness, and real Crew agents add provider-specific behaviour not exercised by the deterministic backend.

In [ ]:
pattern_names = [
    "prompt_chaining",
    "routing",
    "parallelisation",
    "react",
    "planner_executor",
    "critic_reviser",
    "orchestrator_worker",
]
flow_results = {}
with contextlib.redirect_stdout(io.StringIO()):
    for pattern in pattern_names:
        flow_results[pattern] = await MatchedPatternFlow().kickoff_async(
            inputs={"pattern": pattern}
        )
pattern_evaluations = {
    "prompt_chaining": flow_results["prompt_chaining"]["stop"] == "criteria_met",
    "routing": len(flow_results["routing"]["results"]) == 2,
    "parallelisation": flow_results["parallelisation"]["source_ids"]
    == ["food-waste-001", "food-waste-002"],
    "react": flow_results["react"]["steps"] <= 2,
    "planner_executor": flow_results["planner_executor"]["stop"] == "criteria_met",
    "critic_reviser": flow_results["critic_reviser"]["revisions"] == 1,
    "orchestrator_worker": flow_results["orchestrator_worker"]["worker_count"] == 2,
}
assert all(pattern_evaluations.values()), pattern_evaluations
{
    "framework": "CrewAI Flow",
    "evaluation": pattern_evaluations,
    "traces": {k: v["trace"] for k, v in flow_results.items()},
}